# kelly-lab: why betting too big ruins you even when you have an edge

This notebook is the demonstration surface for the repo. It is meant to be read
top to bottom in about ten minutes by someone who has never seen the Kelly
criterion.

Right now it only checks that the environment is set up correctly. The content
arrives step by step, following `TODO.md`:

1. wealth paths for several staked fractions, same random seed
2. the optimal fraction found numerically, then checked against the formula
3. the growth rate curve, and where it crosses back below zero
4. mean against median final wealth

## Setup check

If the cell below fails, the package is not installed. From the repo root:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
pip install -e .
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import kelly

print("kelly", kelly.__version__)
print("numpy", np.__version__)

## The bet, and the size

A coin that pays 2 to 1 on a fair flip. The edge is unambiguous: half a
unit of profit for every unit staked. The only open question is how much
of the bankroll to put on each flip.

In [ ]:
from kelly.simulate import BetSpec, final_log_wealth, ruin_probability

bet = BetSpec(p=0.5, b=2.0)
print(f"edge per unit staked: {bet.edge:+.2f}")

Same bet, same edge, eight stake sizes. `g` is the average log growth per
flip, and the last column is the share of runs that end having lost 99% of
the bankroll.

In [ ]:
HORIZON, PATHS, SEED = 500, 50_000, 20260824

print(f"{'fraction':>9} {'g per flip':>11} {'wiped out':>10}")
for f in (0.05, 0.10, 0.25, 0.40, 0.60, 0.80):
    g = final_log_wealth(bet, f, HORIZON, n_paths=PATHS, seed=SEED).mean() / HORIZON
    ruin = ruin_probability(bet, f, HORIZON, n_paths=PATHS, seed=SEED)
    print(f"{f:9.2f} {g:11.5f} {ruin:9.1%}")

Growth peaks somewhere in the middle and turns negative well before the
stake reaches the whole bankroll, even though every single flip still has
the same positive edge. Finding that peak, and plotting the curve, are the
next steps in `TODO.md`.